In [25]:
# Cell 1: Install dependencies and mount Drive
!pip install timm huggingface_hub openslide-python h5py opencv-python-headless
!apt-get install -y openslide-tools

from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/hb-1968/prame-predict.git

  Reading patches:   1%|▏         | 414/31765 [24:10<30:31:06,  3.50s/patch]


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openslide-tools is already the newest version (3.4.1+dfsg-5build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fatal: destination path 'prame-predict' already exists and is not an empty directory.


In [26]:
# Cell 2: HuggingFace login
from huggingface_hub import login
login()

## Imports and Tiling Module

Load all dependencies and import the optimized tiling pipeline from the repo. This cell also sets `float32_matmul_precision` to `medium` for faster matmuls on supported hardware.

In [27]:
import requests
import shutil
import threading
import queue
import numpy as np
import pandas as pd
import torch
torch.set_float32_matmul_precision('medium')
import timm
import h5py
import openslide
from pathlib import Path
from tqdm import tqdm
from importlib.machinery import SourceFileLoader
from concurrent.futures import ThreadPoolExecutor

tile_module = SourceFileLoader("tile", "prame-predict/02_tile_wsi.py").load_module()
tile_slide = tile_module.tile_slide
_close_thread_handles = tile_module._close_thread_handles

## Model Loaders

Functions to load UNI (ViT-Large, 1024-dim) and CONCH (ViT-Base, 768-dim) from HuggingFace. Weights are cached after the first download.

In [28]:
def load_uni():
    """Load UNI (ViT-Large, 1024-dim)."""
    from huggingface_hub import hf_hub_download
    model = timm.create_model("vit_large_patch16_224", init_values=1e-5,
                              num_classes=0, pretrained=False)
    ckpt = hf_hub_download(repo_id="MahmoodLab/uni", filename="pytorch_model.bin")
    model.load_state_dict(torch.load(ckpt, map_location="cpu", weights_only=True), strict=True)
    return model


def load_conch():
    """Load CONCH visual encoder (ViT-Base, 768-dim)."""
    from huggingface_hub import hf_hub_download
    model = timm.create_model("vit_base_patch16_224", num_classes=0, pretrained=False)
    ckpt = hf_hub_download(repo_id="MahmoodLab/conch", filename="pytorch_model.bin")
    state_dict = torch.load(ckpt, map_location="cpu", weights_only=True)
    vision_keys = {k.replace("visual.", ""): v for k, v in state_dict.items() if k.startswith("visual.")}
    model.load_state_dict(vision_keys if vision_keys else state_dict, strict=False)
    return model

## Preprocessing and Feature Extraction

Vectorized preprocessing bypasses per-patch PIL transforms. The `BatchPrefetcher` prepares the next batch in a background thread while the GPU processes the current one. Inference uses float16 via `torch.amp.autocast`.

In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


def preprocess_batch(batch_np):
    """Vectorized: uint8 (N,224,224,3) -> normalized float16 tensor on GPU."""
    batch = torch.from_numpy(batch_np).permute(0, 3, 1, 2)
    batch = batch.to(dtype=torch.float16).div_(127.5).sub_(1.0)
    return batch.to(device, non_blocking=True)


class BatchPrefetcher:
    """Prepare next batch in background thread while GPU processes current one."""
    def __init__(self, patches, batch_size):
        self._patches = patches
        self._bs = batch_size
        self._n = len(patches)
        self._q = queue.Queue(maxsize=4)
        threading.Thread(target=self._produce, daemon=True).start()

    def _produce(self):
        for i in range(0, self._n, self._bs):
            self._q.put(preprocess_batch(np.array(self._patches[i:i + self._bs])))
        self._q.put(None)

    def __iter__(self):
        while (b := self._q.get()) is not None:
            yield b.to('cuda', non_blocking=True)


def extract_features(patches, model_name):
    """Run model on patches, return (N, feat_dim) float16 array."""
    n = len(patches)
    num_batches = (n + BATCH_SIZE_GPU - 1) // BATCH_SIZE_GPU
    features = None
    write_idx = 0

    with torch.inference_mode():
        for batch in tqdm(BatchPrefetcher(patches, BATCH_SIZE_GPU),
                          total=num_batches, desc=f"  {model_name}", leave=False):
            with torch.amp.autocast("cuda"):
                feats = model(batch.float())
            feats_np = feats.cpu().numpy().astype(np.float16)
            if features is None:
                features = np.empty((n, feats_np.shape[1]), dtype=np.float16)
            features[write_idx:write_idx + len(feats_np)] = feats_np
            write_idx += len(feats_np)

    return features


def save_h5(path, features, coords, model_name, slide_name):
    """Save features as compressed HDF5."""
    with h5py.File(path, "w") as f:
        f.create_dataset("features", data=features, compression="gzip", compression_opts=4)
        f.create_dataset("coords", data=coords, compression="gzip", compression_opts=4)
        f.attrs["model"] = model_name
        f.attrs["slide_name"] = slide_name
        f.attrs["num_patches"] = features.shape[0]
        f.attrs["feature_dim"] = features.shape[1]

Device: cuda


## Download Helper

Uses a persistent `requests.Session` with connection pooling (16 connections) and 1MB chunk reads. 16 parallel threads download slides from the GDC API.

In [30]:
_session = requests.Session()
_session.mount("https://", requests.adapters.HTTPAdapter(
    pool_connections=16, pool_maxsize=16, max_retries=0
))

# Thread-safe counter for download progress
_dl_lock = threading.Lock()
_dl_done = 0
_dl_bytes = 0


def download_slide(row, pbar, max_retries=3):
    """Download a single slide from GDC with connection reuse and progress tracking."""
    global _dl_done, _dl_bytes
    local_path = local_wsi / row["file_name"]
    if local_path.exists():
        size = local_path.stat().st_size
        with _dl_lock:
            _dl_done += 1
            _dl_bytes += size
            pbar.set_postfix_str(f"{_dl_bytes / 1e9:.1f} GB", refresh=False)
            pbar.update(1)
        return local_path
    for attempt in range(max_retries):
        try:
            url = f"https://api.gdc.cancer.gov/data/{row['file_id']}"
            resp = _session.get(url, stream=True, timeout=300)
            resp.raise_for_status()
            size = 0
            with open(local_path, "wb") as f:
                for chunk in resp.iter_content(chunk_size=1048576):
                    f.write(chunk)
                    size += len(chunk)
            with _dl_lock:
                _dl_done += 1
                _dl_bytes += size
                pbar.set_postfix_str(f"{_dl_bytes / 1e9:.1f} GB", refresh=False)
                pbar.update(1)
            return local_path
        except Exception as e:
            print(f"  Retry {attempt+1}/{max_retries}: {e}")
            if local_path.exists():
                local_path.unlink()
    print(f"  FAILED: {row['file_name']}")
    with _dl_lock:
        pbar.update(1)
    return None

## Configuration

Pipeline parameters. Uses T4 GPU (~1.67 units/hr). Both UNI and CONCH fit in ~39 of 100 compute units. Tiles are reused across models to avoid redundant work.

In [35]:
manifest = pd.read_csv("prame-predict/data/expression/slide_manifest.csv")
drive_base = Path("/content/drive/MyDrive/prame-predict/embeddings")
local_wsi = Path("/content/wsi_batch")
local_tiles = Path("/content/tiles_tmp")
local_wsi.mkdir(exist_ok=True)
local_tiles.mkdir(exist_ok=True)

BATCH_SIZE_GPU = 128      # patches per forward pass
MAX_PATCHES = 80000       # random sample cap per slide
DOWNLOAD_BATCH = 50       # slides per download batch
DOWNLOAD_WORKERS = 16     # parallel download threads
MODELS = ["uni", "conch"] # both models in one session

print(f"Manifest: {len(manifest)} slides")
print(f"Embeddings: {drive_base}")

Manifest: 200 slides
Embeddings: /content/drive/MyDrive/prame-predict/embeddings


In [34]:
import shutil
from pathlib import Path

for f in Path("/content/wsi_batch").glob("*.svs"):
    f.unlink()
for d in Path("/content/tiles_tmp").iterdir():
    if d.is_dir():
        shutil.rmtree(d)
print("Cleaned!")

Cleaned!


## Run Pipeline

For each model: load weights, filter to unprocessed slides, download in batches of 75, tile + extract + save. Tiles are kept between models so each slide is only tiled once. Resumable â€” skips slides with existing `.h5` files.

In [36]:
for model_name in MODELS:
    print(f"\n{'#'*60}")
    print(f"# MODEL: {model_name.upper()}")
    print(f"{'#'*60}")

    # Load model
    if model_name == "uni":
        model = load_uni()
    else:
        model = load_conch()
    model.eval()
    model = model.to(device)

    emb_dir = drive_base / model_name
    emb_dir.mkdir(parents=True, exist_ok=True)

    # Filter to unprocessed slides
    remaining = []
    for _, row in manifest.iterrows():
        emb_path = emb_dir / row["file_name"].replace(".svs", ".h5")
        if not emb_path.exists():
            remaining.append(row)
    remaining = pd.DataFrame(remaining)
    print(f"Slides remaining: {len(remaining)} / {len(manifest)}")

    if len(remaining) == 0:
        print("All done for this model, skipping")
        del model
        torch.cuda.empty_cache()
        continue

    # Process in download batches
    for batch_start in range(0, len(remaining), DOWNLOAD_BATCH):
        batch = remaining.iloc[batch_start:batch_start + DOWNLOAD_BATCH]
        batch_end = min(batch_start + DOWNLOAD_BATCH, len(remaining))
        print(f"\nBatch: slides {batch_start+1}-{batch_end} of {len(remaining)}")

        # Download in parallel with progress bar
        _dl_done = 0
        _dl_bytes = 0
        with tqdm(total=len(batch), desc="  Downloading", unit="slide") as pbar:
            with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
                futs = {executor.submit(download_slide, row, pbar): row["file_name"]
                        for _, row in batch.iterrows()}
                for f in futs:
                    try: f.result()
                    except Exception as e: print(f"  Download error: {e}")

        # Process each slide
        for _, row in batch.iterrows():
            slide_name = row["file_name"]
            local_path = local_wsi / slide_name
            emb_path = emb_dir / slide_name.replace(".svs", ".h5")

            if emb_path.exists():
                continue
            if not local_path.exists():
                print(f"  {slide_name}: download failed, skipping")
                continue

            print(f"\n  {slide_name}")
            slide_out = local_tiles / slide_name.replace(".svs", "")
            patches_path = slide_out / "patches.npy"

            # Tile only if not already tiled (reuse across models)
            if not patches_path.exists():
                slide_out.mkdir(parents=True, exist_ok=True)
                num_patches, coords = tile_slide(
                    local_path, slide_out, workers=4, max_patches=MAX_PATCHES
                )
                np.save(slide_out / "coords.npy", np.array(coords))
                _close_thread_handles()
            else:
                coords = np.load(slide_out / "coords.npy")
                num_patches = len(coords)

            if num_patches == 0:
                if slide_out.exists():
                    shutil.rmtree(slide_out)
                continue

            # Extract features
            patches = np.load(patches_path)
            features = extract_features(patches, model_name)
            del patches

            # Save to Drive
            save_h5(emb_path, features, np.array(coords), model_name, slide_name)
            print(f"  Saved {features.shape}")
            del features

            # After save_h5, add:
            if slide_out.exists():
              shutil.rmtree(slide_out)

        # Cleanup WSIs after each batch (tiles kept for reuse across models)
        for f in local_wsi.glob("*.svs"):
            f.unlink()
        # Cleanup tiles only after last model
        if model_name == MODELS[-1]:
            for d in local_tiles.iterdir():
                if d.is_dir():
                    shutil.rmtree(d)

    # Free GPU memory before loading next model
    del model
    torch.cuda.empty_cache()

# Final cleanup
for f in local_wsi.glob("*.svs"):
    f.unlink()
if local_tiles.exists():
    shutil.rmtree(local_tiles, ignore_errors=True)
    local_tiles.mkdir(exist_ok=True)

print(f"\nAll done! Embeddings saved to {drive_base}")


############################################################
# MODEL: UNI
############################################################
Slides remaining: 101 / 200

Batch: slides 1-50 of 101


  Downloading:   0%|          | 0/50 [00:00<?, ?slide/s]

  Retry 1/3: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


  Downloading:  88%|████████▊ | 44/50 [09:42<01:19, 13.24s/slide, 82.5 GB]


KeyboardInterrupt: 